
## Chapter 6: Naive Bayes algorithm

### Importing the necessary packages

In [2]:
from matplotlib import pyplot as plt
import numpy as np
import random
import pandas as pd

### Download dataset and Unzip it

In [3]:
#!/bin/bash
!curl -L -o /content/spam-filter.zip\
  https://www.kaggle.com/api/v1/datasets/download/karthickveerakumar/spam-filter

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2924k  100 2924k    0     0  5463k      0 --:--:-- --:--:-- --:--:-- 5463k


In [4]:
#!/bin/bash
!unzip /content/spam-filter.zip -d /content/

Archive:  /content/spam-filter.zip
  inflating: /content/emails.csv     


### Looking at Dataset table

In [5]:
import pandas
emails = pandas.read_csv('emails.csv')

In [6]:
display(emails.head(10))

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
5,"Subject: great nnews hello , welcome to medzo...",1
6,Subject: here ' s a hot play in motion homela...,1
7,Subject: save your money buy getting this thin...,1
8,Subject: undeliverable : home based business f...,1
9,Subject: save your money buy getting this thin...,1


### Data preprocessing

In [11]:
def process_email(text):
  return list(set(text.split()))

In [12]:
emails['words'] = emails['text'].apply(process_email)

In [13]:
display(emails.head(10))

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[information, not, structure, products, promis..."
1,Subject: the stock trading gunslinger fanny i...,1,"[morristown, optima, chronography, herald, pen..."
2,Subject: unbelievable new homes made easy im ...,1,"[visit, homeowner, easy, we, your, this, advan..."
3,Subject: 4 color printing special request add...,1,"[goldengraphix, information, or, (, this, soli..."
4,"Subject: do not have money , get software cds ...",1,"[cds, d, it, not, best, are, old, tradgedies, ..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[inexpiable, cosmopolitan, nice, r, op, confid..."
6,Subject: here ' s a hot play in motion homela...,1,"[constant, act, u, automation, limit, seven, b..."
7,Subject: save your money buy getting this thin...,1,"[any, effect, exactiy, cannot, than, it, advan..."
8,Subject: undeliverable : home based business f...,1,"[not, ims, original, 000, ;, 8, +, subject, 25..."
9,Subject: save your money buy getting this thin...,1,"[any, effect, exactiy, cannot, than, it, advan..."


### Finding the priors

In [14]:
prob_of_spam = sum(emails['spam'])/len(emails)
prob_of_ham = 1 - prob_of_spam

print('Probability of spam: ', prob_of_spam)
print('Probability of ham: ', prob_of_ham)

Probability of spam:  0.2388268156424581
Probability of ham:  0.7611731843575419


### Finding the posteriors with Bayes theorem

In [15]:
model = {}

for index, email in emails.iterrows():
  for word in email['words']:
    if word not in model:
      model[word] = {'spam': 1, 'ham': 1}
    if email['spam']:
      model[word]['spam'] += 1
    else:
      model[word]['ham'] += 1

In [30]:
# show 10 top in model dict
print(dict(list(model.items())[0:10]))

{'identity': {'spam': 81, 'ham': 4}, 'promise': {'spam': 37, 'ham': 19}, 'our': {'spam': 623, 'ham': 1492}, 't': {'spam': 334, 'ham': 748}, 'market': {'spam': 114, 'ham': 491}, 'and': {'spam': 989, 'ham': 3569}, 'ciear': {'spam': 10, 'ham': 1}, 'that': {'spam': 533, 'ham': 2471}, 'no': {'spam': 395, 'ham': 624}, 'naturally': {'spam': 9, 'ham': 8}}


In [16]:
model['lottery']

{'spam': 9, 'ham': 1}

In [17]:
model['sale']

{'spam': 39, 'ham': 42}

### Using the model to make predictions



In [18]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]['spam']
    num_ham_with_word = model[word]['ham']
    return 1.0*num_spam_with_word/(num_spam_with_word + num_ham_with_word)

In [19]:
predict_bayes('lottery')

0.9

In [20]:
predict_bayes('sale')

0.48148148148148145

### Naive Bayes algorithm

In [21]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails['spam'])
    num_ham = total - num_spam
    email = email.lower()
    words = set(email.split())
    spams = [1.0]
    hams = [1.0]
    for word in words:
        if word in model:
            spams.append(model[word]['spam']/num_spam*total)
            hams.append(model[word]['ham']/num_ham*total)
    prod_spams = np.long(np.prod(spams)*num_spam)
    prod_hams = np.long(np.prod(hams)*num_ham)
    return prod_spams/(prod_spams + prod_hams)

In [22]:
predict_naive_bayes('lottery sale')

np.float64(0.9638144992048691)

In [23]:
predict_naive_bayes('Hi mom how are you')

np.float64(0.12554358867164464)

In [24]:
predict_naive_bayes('Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd')

np.float64(0.12554358867164464)

In [26]:
predict_naive_bayes('meet me at the lobby of the hotel at nine am')

/tmp/ipython-input-612803891.py:13: RuntimeWarning: invalid value encountered in cast
  prod_spams = np.long(np.prod(spams)*num_spam)
/tmp/ipython-input-612803891.py:14: RuntimeWarning: invalid value encountered in cast
  prod_hams = np.long(np.prod(hams)*num_ham)
/tmp/ipython-input-612803891.py:15: RuntimeWarning: overflow encountered in scalar add
  return prod_spams/(prod_spams + prod_hams)
/tmp/ipython-input-612803891.py:15: RuntimeWarning: divide by zero encountered in scalar divide
  return prod_spams/(prod_spams + prod_hams)


np.float64(-inf)

In [27]:
predict_naive_bayes('enter the lottery to win three million dollars')

/tmp/ipython-input-612803891.py:13: RuntimeWarning: invalid value encountered in cast
  prod_spams = np.long(np.prod(spams)*num_spam)
/tmp/ipython-input-612803891.py:14: RuntimeWarning: invalid value encountered in cast
  prod_hams = np.long(np.prod(hams)*num_ham)
/tmp/ipython-input-612803891.py:15: RuntimeWarning: overflow encountered in scalar add
  return prod_spams/(prod_spams + prod_hams)
/tmp/ipython-input-612803891.py:15: RuntimeWarning: divide by zero encountered in scalar divide
  return prod_spams/(prod_spams + prod_hams)


np.float64(-inf)

In [28]:
predict_naive_bayes('buy cheap lottery easy money now')

np.float64(0.999973472265966)

In [29]:
predict_naive_bayes('Grokking Machine Learning by Luis Serrano')

np.float64(0.4197107645488719)

In [30]:
predict_naive_bayes('asdfgh')

np.float64(0.2388268156424581)

## Refactored Vaive Bayes model


- Naive Bayes multi-class classifier inplemented from scratch.
- Handles zero frequency corrections/smoothing.

In [43]:
import collections
import numpy as np
import pandas as pd
import pathlib

### 1. Calculating prior probabilities

Priors are the probabilities of seeing the classifications from just the labeled data

In [35]:
# Helpers (Priors) ========================================

def calculate_frequency_average(series):
    """ Calculates probabilites of occurences of each label out of the entire set """
    try:
        series_averages = series.value_counts() / len(series)
        return series_averages.to_dict()
    except ZeroDivisionError as exception:
        raise exception

### 2. Training a Naive Bayes model

We need to calculate the text's word frequencies in order to train the model. Our plan is to write a dictionary that records every word, and calculate its pair of occurrences in spam and ham.

Sometimes, if we train on new text, we may see a word that we haven't seen before. In order for the math to check out (avoid dividing by zero), we may have to add a tiny number, and we'll use the whole text to cook up this number.

In [36]:
# Helpers (Dictionaries) =========================================

def merge_dicts(dict1, dict2):
    dictionary = dict1.copy()
    dictionary.update(dict2)
    return dictionary

def construct_frequency_dict_from_series(series_text):
    """ Converts text to lower-case then returns dictionary of word counts """
    series_counts = (
        series_text
        .str.lower()
        .str.split() # string -> list
        .explode() # produces row for each item in list
        .value_counts()
    )

    return series_counts.to_dict()

def construct_frequency_dict_from_strings(list_strings):
    """ Converts text to lower-case then returns list of unique words """
    string = " ".join(list_strings)
    string = string.lower()
    list_words = string.split()

    return dict(collections.Counter(list_words))

# Model =========================================

def calculate_labeled_frequencies(dict_frequencies_text, dataframe, column_label, column_words):
    """
    Constructs a frequency dictionary for list of words.
    Uses a processed dataframe with list words column.

    Handles zero frequency occurences by adding n(w) / total,
    in which n(w) is the number of occurences of the word
    across all text, and total is the total number of words
    in the text.

    Parameter
    ----------
    dict_frequencies_text = { word : n(word) }
    dataframe[column_words] = series of list of strings

    """
    list_labels = dataframe[column_label].unique()
    total = sum(dict_frequencies_text.values())

    # Doing it this way avoids copy errors with nested dictionaries
    model = {}
    for word in dict_frequencies_text.keys():
        model.setdefault(word, {label : 0 for label in list_labels })

    # Split label column into groups so we can count them directly
    group_labels = dataframe.groupby(column_label)
    for label, label_df in group_labels:
        for list_words in label_df[column_words]:
            for word in list_words:
                model[word][label] += 1

    # Handles the zero frequency offset
    for word, dict_frequency in model.items():
        offset = max(dict_frequencies_text[word] / total, 1E-8)
        for key in dict_frequency.keys():
            dict_frequency[key] += offset

    return model

### 3. Using the model to make predictions

In [41]:
def predict_bayes(word, label, dict_frequencies):
    """
    Doesn't use the naive assumption.
    likelihood = (num labeled text with word) / sum(num labeled text with word for all labels)
               = P(A | Event_j) / sum ( P (A | Event_i) )
    """
    label_count = dict_frequencies[word][label]
    all_label_counts = sum(dict_frequencies[word].values())

    try:
        return label_count / all_label_counts
    except ZeroDivisionError:
        return 0

In [40]:
# Helpers (Probabilities) =============================

def calculate_list_product(list_):
    """ Slightly faster to work on arrays than directly with list """
    return mp.array(list_).prod()

# Naive Bayes Classifier ==============================

def setup_naive_bayes(dict_frequencies_new_text, dataframe, column_label, column_words, column_text):
    """
    Adds new text to the model so it can be used to make new predictions.
    This is mostly just an accumulation of the previous cells.
    """

    dict_frequencies_whole_text = construct_frequency_dict_from_series(dataframe[column_text])

    dict_model = calculate_labeled_frequencies(
        merge_dicts(dict_frequencies_whole_text, dict_frequencies_new_text),
        dataframe,
        column_label,
        column_words)

    return dict_model

def calculate_naive_bayes(list_words, dict_frequencies, series_labels):
    list_labels = series_labels.unique()
    counts_label = series_labels.value_counts()
    total = len(series_labels)

    # Calculate the probabilty of given label for each word, then accumulates.
    dict_naive_bayes = { label : 1 for label in list_labels }
    for word in list_words:
        for label in list_labels:
            probability = dict_frequencies[word][label] / counts_label[label]
            if probability == 0:
                print(word)
            dict_naive_bayes[label] *= (probability * total)

    # Multiply by the total number of elements for each label
    for label in list_labels:
        dict_naive_bayes[label] *= counts_label[label]

    return dict_naive_bayes

def predict_naive_bayes(document, label_to_predict, dict_frequencies, series_labels):
    """ Uses the naive assumption to predict on a given document """

    # words
    document = document.lower()
    words = set(document.split())

    dict_naive_bayes = calculate_naive_bayes(words, dict_frequencies, series_labels)

    numerator = dict_naive_bayes[label_to_predict]
    denominator = sum(dict_naive_bayes.values())

    return numerator/denominator

### Predicting Ham vs Spam emails

### 1. Imports and pre-processing data

We load the data into a Pandas dataframe, then we preprocess it by adding a column with the (non-repeated) lowercase words in the email.

In [45]:
# Environment variables
dir_path = pathlib.Path.cwd()
name_dataset = "content/emails.csv"

column_emails = "text"
column_words = "words"
column_label = "spam"

# Read dataset
emails = pandas.read_csv(dir_path.parents[0] / name_dataset)
emails[:10]

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
5,"Subject: great nnews hello , welcome to medzo...",1
6,Subject: here ' s a hot play in motion homela...,1
7,Subject: save your money buy getting this thin...,1
8,Subject: undeliverable : home based business f...,1
9,Subject: save your money buy getting this thin...,1


In [46]:
# Helpers (Preprocess) =========================================

def split_string_into_unique_words(string):
    return list(set(string.split()))

def process_series_email(series_text):
    """ Converts text to lower-case then returns list of unique words """
    series_words = series_text.copy() # copies original series
    series_words = series_words.str.lower()
    series_words = series_words.apply(split_string_into_unique_words)

    return series_words

In [47]:
emails[column_words] = process_series_email(emails[column_emails])
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[information, not, structure, products, promis..."
1,Subject: the stock trading gunslinger fanny i...,1,"[morristown, optima, chronography, subject:, p..."
2,Subject: unbelievable new homes made easy im ...,1,"[visit, homeowner, subject:, easy, we, your, t..."
3,Subject: 4 color printing special request add...,1,"[goldengraphix, information, subject:, or, (, ..."
4,"Subject: do not have money , get software cds ...",1,"[cds, d, subject:, it, not, best, are, old, tr..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[inexpiable, cosmopolitan, nice, r, subject:, ..."
6,Subject: here ' s a hot play in motion homela...,1,"[constant, act, u, automation, limit, seven, b..."
7,Subject: save your money buy getting this thin...,1,"[any, effect, exactiy, cannot, subject:, than,..."
8,Subject: undeliverable : home based business f...,1,"[not, ims, original, 000, ;, 8, +, subject, 25..."
9,Subject: save your money buy getting this thin...,1,"[any, effect, exactiy, cannot, subject:, than,..."


### 2. Calculate the priors

Our label column is boolean, with spam being 1 and ham being 0. Let's calculate the probabilities of seeing a ham or spam email from just the labeled data.

In [48]:
label_spam = 1
label_ham = 0

# meta data
num_emails = len(emails)
counts_label = emails[column_label].value_counts()
num_spam = counts_label[label_spam]
print(counts_label)

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)

# Calculating the prior probability an email is spam.
dict_priors = calculate_frequency_average(emails[column_label])
print("Probability of spam:", dict_priors[label_spam])

spam
0    4360
1    1368
Name: count, dtype: int64
Number of emails: 5728
Number of spam emails: 1368
Probability of spam: 0.2388268156424581


### 3. Training the model

We'll calculate word frequencies based on the whole text, then train the model by calculating the frequencies for each word in each label.

In [49]:
dict_frequencies_whole_text = construct_frequency_dict_from_series(
    emails[column_emails])
dict_model = calculate_labeled_frequencies(
    dict_frequencies_whole_text, emails, column_label, column_words)

In [50]:
# Some examples (1 is spam, and 0 is ham)
print(dict_model['lottery'])
print(dict_model['sale'])
print(dict_model['already'])

{np.int64(1): 8.000010682682143, np.int64(0): 1.0682682143736557e-05}
{np.int64(1): 38.00005661821536, np.int64(0): 41.00005661821536}
{np.int64(1): 64.0002382238118, np.int64(0): 317.0002382238118}


### 4. Using the model to make predictions

We can see the probability a word is associated with spam given our data. We can also add new words to our model by calculating their word frequencies.

In [51]:
print(predict_bayes('lottery', label_spam, dict_model))
print(predict_bayes('sale', label_spam, dict_model))
print(predict_bayes('already', label_spam, dict_model))

0.9999986646682983
0.4810126854437434
0.1679794178226178


In [52]:
list_emails = [
    "lottery sale",
    "Hi mom how are you",
    "Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd",
    "meet me at the lobby of the hotel at nine am",
    "enter the lottery to win three million dollars",
    "buy cheap lottery easy money now",
    "buy cheap lottery easy money"
    "Grokking Machine Learning by Luis Serrano",
    "asdfgh"]

# Adding new words to our dictionary
dict_frequencies_new_words = construct_frequency_dict_from_strings(list_emails)
dict_model = setup_naive_bayes(
    dict_frequencies_new_words, emails, column_label, column_words, column_emails)
cout = "Probability email is spam: "
for email in list_emails:
    print(cout, predict_naive_bayes(email, label_spam, dict_model, emails[column_label]))

Probability email is spam:  0.9999999005387217
Probability email is spam:  0.09520065375112552
Probability email is spam:  0.2511282162504502
Probability email is spam:  3.410728699614588e-11
Probability email is spam:  0.9999999987178892
Probability email is spam:  0.9999999999343989
Probability email is spam:  0.9999999996071451
Probability email is spam:  0.5000000000000001


### 5. Do our results make sense?

The "Grokking Machine Learning by Luis Serrano" classification was surprising. Or was it? Let's check how often a word like "serrano" appears in spam emails.

In [53]:
print(dict_model['serrano'])
print(predict_bayes('serrano', label_spam, dict_model))

{np.int64(1): 1.0000005876034477, np.int64(0): 5.876034475869477e-07}
0.9999994123972429


Hmm, that seeems pretty high. But, if we look closer at the training data, the following email was labaled spam and has "serrano"!



> Subject: important announcement : your application was approved we tried to contact you last week about refinancing your home at a lower rate . i would like to inform you know that you have been pre - approved . here are the results : * account id : [ 987 - 528 ] * negotiable amount :
 690 , 043 * rate : 3 . 70 % - 5 . 68 % please fill out this quick form and we will have a broker contact you as soon as possible . regards , shannon serrano senior account manager lyell national lenders , llc . database deletion : www . lend - bloxz . com / r . php



Talk about bad luck. This highlights the importance of cleaning the data before you train!